# PyVista 的数据集

PyVista 是让三维可视化与网格操作变简单的库，把数据放在正确的结构里，是可视化的第一步。

这一章说明 PyVista 数据集的基本，弄清了数据的结构，操作三维数据、用图确认解析结果都会变得轻松。

In [1]:
from helper.python import print_subclasses
import helper.magics
import pyvista as pv
import numpy as np
from helper.pyvista import plotter_to_iframe, plot_point_indices

## 本章要学的内容

PyVista 按数据的种类准备了几个数据集，而数据集都由点(Point)与数据(Data)两部分组成。

点之间可能连着、也可能不连着，几个点合在一起就成了单元(Cell)；点的连接方式既可以显式给出，也可以隐式决定。数据有标量与向量两种，既可能挂在点上，也可能挂在单元上。

这一章一边依次运行例子，一边一点点看清数据集的结构。

* `ImageData`：等间隔的三维格子
* `RectilinearGrid`：间隔不均匀的格子
* `StructuredGrid`：逐个决定点的格子
* `PolyData`：点、线、多边形的面

In [1]:
print_subclasses(pv.DataSet, as_html=True, height=300)


## ImageData

最好懂的数据集是 `ImageData`：它表示二维或三维的图像，可以简单看成数组。

数组里放着数据，点位于正交的等间隔格子上，所以不必决定点的坐标；点之间的连接也由数组的位置决定，是隐式的。

下面的程序做出 `ImageData`，决定 `spacing`、`origin`、`dimensions`。

In [3]:
img = pv.ImageData(
    spacing=(0.1, 0.1, 0.1), origin=(0.1, 0.2, 0.3), dimensions=(3, 4, 5)
)
img

ImageData (0x21669c9cbe0)
  N Cells:      24
  N Points:     60
  X Bounds:     1.000e-01, 3.000e-01
  Y Bounds:     2.000e-01, 5.000e-01
  Z Bounds:     3.000e-01, 7.000e-01
  Dimensions:   3, 4, 5
  Spacing:      1.000e-01, 1.000e-01, 1.000e-01
  N Arrays:     0

`origin` 是三维格子的起始坐标、`spacing` 是 X、Y、Z 轴方向的间隔、`dimensions` 是各轴上的格子数。

用 `img.points` 能取到各点坐标的数组，坐标按 X、Y、Z 轴的顺序递增，下面的程序输出前 6 个点的坐标。

In [4]:
img.points[:6]

array([[0.1, 0.2, 0.3],
       [0.2, 0.2, 0.3],
       [0.3, 0.2, 0.3],
       [0.1, 0.3, 0.3],
       [0.2, 0.3, 0.3],
       [0.3, 0.3, 0.3]])

与各点对应的数据放进 `point_data`，它是 `DataSetAttributes` 的对象。

In [5]:
img.point_data

pyvista DataSetAttributes
Association     : POINT
Active Scalars  : None
Active Vectors  : None
Active Texture  : None
Active Normals  : None
Contains arrays : None

`DataSetAttributes` 里能放多个数组，其中 `active_scalars` 是放着各点标量值的数组。

给 NumPy 数组赋值时，PyVista 会自动做出数组并填进内容：执行下面的程序，`active_scalars` 就从 `None` 变成 `pyvista_ndarray`，这里给每个点放进与下标相同的标量值。

In [6]:
print(img.point_data.active_scalars)  # 没有数据
img.point_data.set_scalars(np.arange(0.0, img.n_points))
print(type(img.point_data.active_scalars))
img.point_data.active_scalars

None
<class 'pyvista.core.pyvista_ndarray.pyvista_ndarray'>


pyvista_ndarray([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.,
                 11., 12., 13., 14., 15., 16., 17., 18., 19., 20., 21.,
                 22., 23., 24., 25., 26., 27., 28., 29., 30., 31., 32.,
                 33., 34., 35., 36., 37., 38., 39., 40., 41., 42., 43.,
                 44., 45., 46., 47., 48., 49., 50., 51., 52., 53., 54.,
                 55., 56., 57., 58., 59.])

`pyvista_ndarray` 继承 NumPy 的 `ndarray`，所以 NumPy 数组的操作全都能用。

In [7]:
img.point_data.active_scalars[:2] = 10, 11
print(img.point_data.active_scalars[:5])

[10. 11.  2.  3.  4.]


`pyvista_ndarray` 也带着 VTK 数组的方法，下面的例子用 `GetName()` 取这个数组的名字。

In [8]:
img.point_data.active_scalars.GetName()

'scalars'

`DataSetAttributes` 能持有多组数组，也能像字典那样用，所以加数据很简单。

In [9]:
img.point_data.keys()

['scalars']

In [10]:
img.point_data["scalars"]

pyvista_ndarray([10., 11.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.,
                 11., 12., 13., 14., 15., 16., 17., 18., 19., 20., 21.,
                 22., 23., 24., 25., 26., 27., 28., 29., 30., 31., 32.,
                 33., 34., 35., 36., 37., 38., 39., 40., 41., 42., 43.,
                 44., 45., 46., 47., 48., 49., 50., 51., 52., 53., 54.,
                 55., 56., 57., 58., 59.])

In [11]:
img.point_data["zerodata"] = np.zeros(img.n_points)
img.point_data.keys()

['scalars', 'zerodata']

用 `del` 指定数组的名字，就能删掉那个数组。

In [12]:
del img.point_data["zerodata"]
img.point_data.keys()

['scalars']

各点的值不只是标量，也可以是向量，向量数组用 `set_vectors()` 加进去。

下面的程序把形状 `(N, 3)` 的 NumPy 数组以 `vectors` 这个名字放进去，其中 `N` 是点的个数，每个点对应一个三维向量。

In [13]:
img.point_data.set_vectors(np.arange(0.0, img.n_points * 3).reshape(-1, 3), "vectors")
img.point_data

pyvista DataSetAttributes
Association     : POINT
Active Scalars  : scalars
Active Vectors  : vectors
Active Texture  : None
Active Normals  : None
Contains arrays :
    scalars                 float64    (60,)                SCALARS
    vectors                 float64    (60, 3)              VECTORS

放进去的数组成为 `active_vectors`，它也能当作 NumPy 数组来用。

In [14]:
img.point_data.active_vectors.shape

(60, 3)

`ImageData` 里点的坐标由 `spacing`、`origin`、`dimensions` 隐式决定，点与单元的关系同样是隐式的。

下面的图表示单元与点的关系：单元是由 8 个点组成的立方体，图里 0 号单元是半透明的灰色立方体。

In [15]:
plotter = pv.Plotter()
plotter.add_mesh(img.extract_all_edges(), color="black")
plotter.add_mesh(
    img.glyph(
        scale=False,
        orient=False,
        factor=0.02,
        geom=pv.Sphere(theta_resolution=6, phi_resolution=6),
    )
)
first_cell = img.extract_cells([0]).extract_surface(pass_pointid=False)

plotter.add_mesh(first_cell, color="gray", opacity=0.5)
plot_point_indices(plotter, first_cell, scale=0.02)
plotter.add_axes()
plotter_to_iframe(plotter)

C:\Users\ruoyu\AppData\Local\Temp\ipykernel_34944\2274479044.py:11: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  first_cell = img.extract_cells([0]).extract_surface(pass_pointid=False)
C:\Users\ruoyu\Documents\notebooks\scipybook\scipybook_cn\helper\pyvista.py:12: PyVistaDeprecationWarning: Plotter.export_vtksz is deprecated. Use plotter.trame.export_vtksz(...) instead. Install via `pip install trame-pyvista`.
  zip_data = plotter.export_vtksz(filename=None)


用 `get_cell()` 能取到 VOXEL(体素)型的 `Cell`，其中 `n_points`、`n_edges`、`n_faces` 是做出那个体素的点、边、面的个数。

In [16]:
cell = img.get_cell(0)
print(cell)
%C cell.n_points; cell.n_edges; cell.n_faces

Cell (0x216760d6bc0)
  Type:        <CellType.VOXEL: 11>
  Linear:      True
  Dimension:   3
  N Points:    8
  N Faces:     6
  N Edges:     12
  X Bounds:    1.000e-01, 2.000e-01
  Y Bounds:    2.000e-01, 3.000e-01
  Z Bounds:    3.000e-01, 4.000e-01
cell.n_points  cell.n_edges  cell.n_faces
-------------  ------------  ------------
8              12            6           


`point_ids` 是做出体素的点的编号列表，`points` 是那些点的坐标。

In [17]:
print(cell.point_ids)
cell.points

[0, 1, 3, 4, 12, 13, 15, 16]


array([[0.1, 0.2, 0.3],
       [0.2, 0.2, 0.3],
       [0.1, 0.3, 0.3],
       [0.2, 0.3, 0.3],
       [0.1, 0.2, 0.4],
       [0.2, 0.2, 0.4],
       [0.1, 0.3, 0.4],
       [0.2, 0.3, 0.4]])

`ImageData` 的 `n_cells` 是数据集里单元的个数，也就是上图中小立方体的个数。

In [18]:
img.n_cells

24

数据集备齐了查点与单元关系的各种方法。

`point_cell_ids()` 返回包含某个点的单元的编号，`cell_neighbors()` 返回某个单元相邻的单元，相邻的判定方式用 `connections` 选，取 `faces` 就是共享面的单元。

In [19]:
print(f"cells of point 3: {img.point_cell_ids(3)}")
print(f"neighbors of cell 0: {img.cell_neighbors(0, connections='faces')}")

cells of point 3: [2, 0]
neighbors of cell 0: [1, 2, 6]


各单元的数据放进 `cell_data`，用法与 `point_data` 相同，这里就不细说了。

In [20]:
img.cell_data

pyvista DataSetAttributes
Association     : CELL
Active Scalars  : None
Active Vectors  : None
Active Texture  : None
Active Normals  : None
Contains arrays : None

## RectilinearGrid

`ImageData` 是最简单的数据集，点全在等间隔的三维格子上，只要知道起始坐标、格子数与间隔，就能算出全部点的坐标。

要表示间隔不齐的格子，就用 `RectilinearGrid`。

In [21]:
x = np.array([0, 3, 9, 15])
y = np.array([0, 1, 5])
z = np.array([0, 2, 3])
rgrid = pv.RectilinearGrid(x, y, z)  #❶
print(f'{rgrid.dimensions = }') #❷
rgrid.point_data["scalars"] = np.arange(0.0, rgrid.n_points, dtype=np.float64)  #❸

rgrid.dimensions = (4, 3, 3)


`RectilinearGrid` 的点也在正交的格子上，只是间隔不齐，所以 X、Y、Z 三个轴各自都要决定格子的面的位置。

❶ 做 `RectilinearGrid` 时用 `x`、`y`、`z` 三个数组决定各轴上垂直于该轴的面位置，点就是这些面相交的地方，❷ 做出来的对象的 `dimensions` 是各轴的面数。

In [22]:
plotter = pv.Plotter()
plotter.add_mesh(rgrid.extract_all_edges(), color="black")
plotter.add_mesh(
    rgrid.glyph(
        scale=False,
        orient=False,
        factor=0.5,
        geom=pv.Sphere(theta_resolution=6, phi_resolution=6),
    )
)
first_cell = rgrid.extract_cells([0]).extract_surface(pass_pointid=False)
plotter.add_mesh(first_cell, color="gray", opacity=0.5)
plot_point_indices(plotter, first_cell, scale=0.5)
plotter.add_axes()
plotter_to_iframe(plotter)

C:\Users\ruoyu\AppData\Local\Temp\ipykernel_34944\1096111344.py:11: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  first_cell = rgrid.extract_cells([0]).extract_surface(pass_pointid=False)
C:\Users\ruoyu\Documents\notebooks\scipybook\scipybook_cn\helper\pyvista.py:12: PyVistaDeprecationWarning: Plotter.export_vtksz is deprecated. Use plotter.trame.export_vtksz(...) instead. Install via `pip install trame-pyvista`.
  zip_data = plotter.export_vtksz(filename=None)


与 `ImageData` 一样，点的编号按 X、Y、Z 轴的顺序递增。

In [23]:
rgrid.points[:10]

array([[ 0.,  0.,  0.],
       [ 3.,  0.,  0.],
       [ 9.,  0.,  0.],
       [15.,  0.,  0.],
       [ 0.,  1.,  0.],
       [ 3.,  1.,  0.],
       [ 9.,  1.,  0.],
       [15.,  1.,  0.],
       [ 0.,  5.,  0.],
       [ 3.,  5.,  0.]])

单元与点的关系也与 `ImageData` 相同。

In [24]:
c = rgrid.get_cell(1)
print("points of cell 1:", repr(c.point_ids))
print(c.points)

points of cell 1: [1, 2, 5, 6, 13, 14, 17, 18]
[[3. 0. 0.]
 [9. 0. 0.]
 [3. 1. 0.]
 [9. 1. 0.]
 [3. 0. 2.]
 [9. 0. 2.]
 [3. 1. 2.]
 [9. 1. 2.]]


## StructuredGrid

`StructuredGrid` 比 `RectilinearGrid` 更自由，点坐标要一个个决定；不过点与单元的关系仍由格子上的排列自动决定。

下面的代码做台形的 `StructuredGrid`：❶ 先用 NumPy 的 `mgrid` 做出 `x`、`y`、`z` 三个数组，形状都是 `(4, 5, 3)`。

In [25]:
x, y, z = np.mgrid[:4.0, :5.0, :3.0]  #❶
x *= (4 - z) / 3  #❷
y *= (4 - z) / 3
sgrid = pv.StructuredGrid(x, y, z) #❸
sgrid.point_data["scalars"] = np.arange(0, sgrid.n_points, dtype=np.float64)

数组 `x` 的 `shape` 就直接成为 `StructuredGrid` 的 `dimensions`。

In [26]:
x.shape, sgrid.dimensions

((4, 5, 3), (4, 5, 3))

数组 `x` 沿第 0 轴变化，所以把它拉成一维时，X 轴的值最先变。

In [27]:
sgrid.points[:10]

pyvista_ndarray([[0.        , 0.        , 0.        ],
                 [1.33333333, 0.        , 0.        ],
                 [2.66666667, 0.        , 0.        ],
                 [4.        , 0.        , 0.        ],
                 [0.        , 1.33333333, 0.        ],
                 [1.33333333, 1.33333333, 0.        ],
                 [2.66666667, 1.33333333, 0.        ],
                 [4.        , 1.33333333, 0.        ],
                 [0.        , 2.66666667, 0.        ],
                 [1.33333333, 2.66666667, 0.        ]])

下面的图是做出的格子的样子，半透明的灰色部分是 0 号单元。

In [28]:
plotter = pv.Plotter()
plotter.add_mesh(sgrid.extract_all_edges(), color="black")
plotter.add_mesh(
    sgrid.glyph(
        scale=False,
        orient=False,
        factor=0.15,
        geom=pv.Sphere(theta_resolution=6, phi_resolution=6),
    )
)
first_cell = sgrid.extract_cells([0]).extract_surface(pass_pointid=False)
plotter.add_mesh(first_cell, color="gray", opacity=0.5)
plot_point_indices(plotter, first_cell, scale=0.2)
plotter.add_axes()
plotter_to_iframe(plotter)

C:\Users\ruoyu\AppData\Local\Temp\ipykernel_34944\1767120552.py:11: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  first_cell = sgrid.extract_cells([0]).extract_surface(pass_pointid=False)


单元的形不是长方体，不过在 PyVista 里这种单元用 `Hexahedron`(六面体)型表示；单元有 `faces` 与 `edges`，能取到做出它的面与边。

In [29]:
c = sgrid.get_cell(2)
print("cell type:", repr(c.type))
print("number_of_faces:", len(c.faces))  # 单元的面数
f = c.faces[0]  # 取第 0 个面
print("face type:", repr(f.type))  # 各面用 Quad 对象表示
print("points of face 0:", repr(f.point_ids))  # 组成第 0 个面的 4 个点的编号
print("edge count of cell:", len(c.edges))  # 单元的边数
e = c.edges[0]  # 取第 0 条边
print("edge type:", repr(e.type))
print("points of edge 0:", repr(e.point_ids))  # 组成第 0 条边的 2 个点的编号

cell type: <CellType.HEXAHEDRON: 12>
number_of_faces: 6
face type: <CellType.QUAD: 9>
points of face 0: [2, 22, 26, 6]
edge count of cell: 12
edge type: <CellType.LINE: 3>
points of edge 0: [2, 3]


用 `StructuredGrid` 能做出任意形状的格子，下面的程序做出如下图所示中间空心的半个圆柱。

先在圆柱坐标里做等间隔的格子，再把各点的坐标转换成直角坐标，具体步骤请自己跟着算一遍确认。

In [30]:
r, theta, z2 = np.mgrid[2:3:3j, -np.pi / 2 : np.pi / 2 : 6j, 0:4:7j]
x2 = np.cos(theta) * r
y2 = np.sin(theta) * r

sgrid2 = pv.StructuredGrid(x2, y2, z2)
sgrid2.point_data["scalars"] = np.arange(0, sgrid2.n_points, dtype=np.float64)

In [31]:
plotter = pv.Plotter()
plotter.add_mesh(sgrid2.extract_all_edges(), color="black")
plotter.add_mesh(
    sgrid2.glyph(
        scale=False,
        orient=False,
        factor=0.15,
        geom=pv.Sphere(theta_resolution=6, phi_resolution=6),
    )
)
plotter.add_axes()
plotter_to_iframe(plotter)

## PolyData

`PolyData` 由点、连接点的线、用点做的多边形的面组成，因为都要自己决定，所以用程序从零做 `PolyData` 比较麻烦。

不过 PyVista 的很多函数都返回 `PolyData`，例如做圆锥的 `Cone` 就像下面这样用。

In [32]:
cone = pv.Cone(resolution=4)
type(cone)

pyvista.core.pointset.PolyData

`PolyData` 的 `points` 是点坐标的数组，为了看清坐标，用 NumPy 的 `array_str()` 输出内容，并用 `suppress_small` 把小值显示成 0。

In [33]:
print(np.array_str(cone.points, suppress_small=True))

[[ 0.5  0.   0. ]
 [-0.5  0.5  0. ]
 [-0.5  0.   0.5]
 [-0.5 -0.5  0. ]
 [-0.5 -0.  -0.5]]


`Cone` 的 `resolution` 设成了 4，所以圆锥的底面是四边形。

看点坐标就知道，底面垂直于 X 轴、位于 `X = -0.5` 的面上，顶点在 `X = 0.5`；点之间的关系由 `faces` 决定。

In [34]:
print(cone.n_cells)  # 圆锥有 5 个面
print(cone.faces)

5
[4 4 3 2 1 3 0 1 2 3 0 2 3 3 0 3 4 3 0 4 1]


`faces` 持有面与点的关系，这个圆锥有 5 个面，所以 `n_cells` 是 5。

`faces` 是一维的整数数组，放着各面的点的编号；因为各面的点数不同，所以把点数也放进去。这个数组要这样读：每一行是一个面，冒号前是那个面的点数、后面是 `points` 里的点的编号。

下面自己来做 `PolyData`，先看一个简单角锥的例子。

❶ 用形状 `(N, 3)` 的 NumPy 数组做出 N 个点的坐标。
❷ 做出表示面与点关系的列表。
❸ 用 `points` 与 `faces` 做出 `PolyData`。
❹ 把各点的数据当作 `scalars` 放进去。

In [35]:
points = np.array(
    [(1, 1, 0), (1, -1, 0), (-1, -1, 0), (-1, 1, 0), (0, 0, 2)], dtype=np.float64
)  # ❶
faces = [4, 0, 1, 2, 3, 3, 4, 0, 1, 3, 4, 1, 2, 3, 4, 2, 3, 3, 4, 3, 0] #❷
p1 = pv.PolyData(points, faces) #❸
p1.point_data["scalars"] = np.arange(0.0, len(p1.points)) #❹

做出来的角锥如下图所示，图里标出了各点的编号。

`PolyData` 的面用 `get_cell()` 取，图里画出了 0 号与 1 号面。

In [36]:
plotter = pv.Plotter()
plotter.add_mesh(p1, show_edges=True)
plotter.add_mesh(
    p1.glyph(
        scale=False,
        orient=False,
        factor=0.15,
        geom=pv.Sphere(theta_resolution=6, phi_resolution=6),
    )
)
plotter.add_mesh(p1.extract_cells([0, 1]), color='gray', opacity=1)
plot_point_indices(plotter, p1, 0.2, index_name='scalars')
plotter.add_axes()
plotter_to_iframe(plotter)

接着取 0 号与 1 号面的点的编号。

In [37]:
print(p1.get_cell(0).point_ids)
print(p1.get_cell(1).point_ids)

[0, 1, 2, 3]
[4, 0, 1]


下面是用 `PolyData` 做半球面的程序：先做球坐标的格子，再转换成直角坐标，其中 `x`、`y`、`z` 是 `(10, 10)` 的二维数组。

❶ 把三个数组合成形状 `(N, 3)` 的数组，❷ 每个面的点数都相同时，面与点的关系就能用二维数组表示：第 0 轴是面的个数、第 1 轴是各面的点数 + 1。❸ 每个面由 4 个点做成，这个二维数组可以直接传给 `PolyData`。

In [38]:
N = 10
a, b = np.mgrid[0 : np.pi : N * 1j, 0 : np.pi : N * 1j]
x = np.sin(a) * np.cos(b)
y = np.sin(a) * np.sin(b)
z = np.cos(a)

points = np.c_[x.ravel('F'), y.ravel('F'), z.ravel('F')]  #❶
faces = np.zeros(((N - 1) ** 2, 5), np.int32)  #❷
t1, t2 = np.mgrid[: (N - 1) * N : N, : N - 1]
faces[:, 0] = 4 #❸
faces[:, 1] = (t1 + t2).ravel()
faces[:, 2] = faces[:, 1] + 1
faces[:, 3] = faces[:, 2] + N
faces[:, 4] = faces[:, 1] + N

p2 = pv.PolyData(points, faces)
p2.point_data["scalars"] = np.arange(0.0, p2.n_points)

下面的图是做出来的半球面。

In [39]:
plotter = pv.Plotter()
plotter.add_mesh(p2, show_edges=True)
plotter.add_axes()
plotter_to_iframe(plotter)

## 小结

* 数据集由点与数据组成，点合在一起就成了单元。
* `ImageData` 是等间隔的格子，坐标由 `origin`、`spacing`、`dimensions` 隐式决定。
* `RectilinearGrid` 是间隔不均的格子，用数组决定各轴的面的位置。
* `StructuredGrid` 逐个决定点的坐标，而点与单元的关系由格子的排列决定。
* `PolyData` 自己决定点、线、面，最好记住 `points` 与 `faces` 的形状。
* 点的数据放进 `point_data`、单元的数据放进 `cell_data`，两者都是 `DataSetAttributes`。